In [1]:
from pathlib import Path
import os
import pandas as pd
import torch


PROD = False

PATH_MODEL = Path(os.environ.get(
    "T2M_MODEL_DEFAULT",
    "path/to/model.pt",  # fallback
))
if not PATH_MODEL.exists():
    raise FileNotFoundError(f"Could not find `{PATH_MODEL}`")
print(f"Set model vocabulary to `{PATH_MODEL}`")

PATH_VOCAB = Path(os.environ.get(
    "T2M_TRAINING_VOCAB",
    "path/to/training_vocab.parquet",  # fallback
))
if not PATH_VOCAB.exists():
    raise FileNotFoundError(f"Could not find `{PATH_VOCAB}`")
print(f"Set training vocabulary to `{PATH_VOCAB}`")

PATH_LOOKUP = Path(os.environ.get(
    "T2M_TRACK_LOOKUP",
    "path/to/track_lookup.parquet",  # fallback
))
if not PATH_LOOKUP.exists():
    raise FileNotFoundError(f"Could not find `{PATH_LOOKUP}`")
print(f"Set track lookup to `{PATH_LOOKUP}`")

# Loading model

In [2]:
device = torch.device('cpu')

ckpt     = torch.load(PATH_MODEL, map_location=device, weights_only=False)
hparams  = ckpt["hparams"]
vocab_ckpt = ckpt["vocab"]                                       # save before del

emb = ckpt["model_state_dict"]["embeddings_in.weight"].clone()   # (V, D), fp32
del ckpt                                                         # frees out-embedding + state dict
emb_norm = emb / emb.norm(dim=1, keepdim=True)

print(f"Embeddings: {emb.shape}  ({emb.element_size() * emb.numel() / 1e9:.2f} GB)")

Embeddings: torch.Size([5412049, 128])  (2.77 GB)


# Loading lookup

In [3]:
vocab = pd.DataFrame(vocab_ckpt)  # track_rowid, track_id — exactly what was used in training

lookup = pd.read_parquet(
    PATH_LOOKUP,
    columns=["track_rowid", "track_name", "artist_name", "track_popularity"],
)
lookup = lookup.merge(vocab, on="track_rowid", how="inner")
print(f"{len(lookup):,} tracks in lookup (should match vocab size {hparams['vocab_size']:,})")
lookup.head()

5,412,049 tracks in lookup (should match vocab size 5,412,049)


,track_rowid,track_name,artist_name,track_popularity,track_id
0,1,The Giver,Chappell Roan,89,0
1,2,Another Life,NOTD,43,1
2,3,Lover Online,NOTD,43,2
3,4,Crash,NOTD,56,3
4,5,I Just Missed A Call,NOTD,51,4


In [4]:
def find_neighbours(
    query: str,
    artist: str | None = None,
    k: int = 10,
    diverse: bool = True,
) -> pd.DataFrame:
    """Find top-k nearest neighbours by cosine similarity.

    `query` is matched case-insensitively against track_name.
    `artist` optionally narrows the match to a specific artist (case-insensitive contains).
    If multiple tracks still match, the most popular one is used.
    If `diverse=True`, at most one track per artist is returned in the results.
    """
    matches = lookup[lookup["track_name"].str.contains(query, case=False, na=False)]
    if artist is not None:
        matches = matches[matches["artist_name"].str.contains(artist, case=False, na=False)]
    if matches.empty:
        desc = f"'{query}'" + (f" by '{artist}'" if artist else "")
        print(f"No tracks matching {desc}")
        return pd.DataFrame()

    row = matches.sort_values("track_popularity", ascending=False).iloc[0]
    tid = int(row["track_id"])
    print(f"Query: {row['track_name']} — {row['artist_name']}  (pop {row['track_popularity']})")

    # topk avoids materialising the full vocab similarity array
    fetch = k * 50 + 1          # overfetch to survive diversity dedup; +1 for the query itself
    sims  = emb_norm[tid] @ emb_norm.T
    top_vals, top_idx = torch.topk(sims, min(fetch, len(sims)))

    sim_map    = dict(zip(top_idx.tolist(), top_vals.tolist()))
    candidates = lookup[lookup["track_id"].isin(sim_map)].copy()
    candidates["similarity"] = candidates["track_id"].map(sim_map)
    candidates = candidates[candidates["track_id"] != tid]
    candidates = candidates.sort_values("similarity", ascending=False)
    if diverse:
        candidates = candidates.drop_duplicates(subset="artist_name")
    return candidates.head(k)[["track_name", "artist_name", "track_popularity", "similarity"]]

In [20]:
queries = [
    ("Michael", "Red House Painters"),
]

for q in queries:
    display(find_neighbours(*q, k=10))

Query: Michael — Red House Painters  (pop 28)


,track_name,artist_name,track_popularity,similarity
2706961,Down Colorful Hill,Red House Painters,31,0.938290
2904528,Ltd,Bluetile Lounge,14,0.859349
2346157,Somewhere (Version 2),Sun Kil Moon,28,0.857916
2787250,Alphabet on the Manhole,Carissa's Wierd,15,0.851536
4595160,Carved Heart,NSB Archive,11,0.844760
3296606,Exhume,Bedhead,14,0.842483
2483397,Corner Store - SOS Demo,Codeine,16,0.842345
2962928,Midnight Willie,David Kauffman,1,0.838251
2540701,Third And South,Birddog,16,0.837674
3990987,R U Still in 2 It?,Mogwai,25,0.836719
